In [ ]:
# %% [markdown]
"""
# Predictive Maintenance with Window-Based Feature Aggregation and Pressure Classification
# (Without Scaling)

**Objective:**  
Build predictive models for:
- **mech_1**: Predict fault (0/1) between sensors A and C.
- **mech_2**: Predict fault (0/1) between sensors C and B.
- **Pressure**: Instead of predicting a continuous value, classify the pressure as:
    - 0 ("low"): Pressure < 4.9 bar,
    - 1 ("high"): Pressure > 5.2 bar,
    - 2 ("normal"): Pressure in [4.9, 5.2] bar.

We compute aggregated features from the "Time series" column using a sliding window (window size = 10)
and split the dataset (with stratification) into training, validation, and test sets.  
This version does NOT apply feature scaling.
"""
# %% 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')


In [3]:
# %% [markdown]
"""
## 1. Data Loading and Preprocessing

Load the four CSV files.  
Each CSV file (without a header) is assumed to have 3 columns:
- **Time series**: Time (milliseconds) of the workflow.
- **Pressure**: Measured pressure (continuous, in bar).
- **Fault**: Binary indicator (0 or 1) that denotes whether a mechanical fault exists.

We add a sensor configuration label to each file:
- **AC** and **CA** for mech_1 prediction,
- **CB** and **BC** for mech_2 prediction.
"""
# %% 
# Adjust file paths as needed.
ac_df = pd.read_csv('C:/Users/Алишер/Desktop/NazarbayevUniversity/DatasetPdM/Class_regres_dataset/AC_Down_to_Sort - Sheet1.csv', header=0)
ca_df = pd.read_csv('C:/Users/Алишер/Desktop/NazarbayevUniversity/DatasetPdM/Class_regres_dataset/CA_Sort_to_Down - Sheet1.csv', header=0)
cb_df = pd.read_csv('C:/Users/Алишер/Desktop/NazarbayevUniversity/DatasetPdM/Class_regres_dataset/CB_Sort_to_Upstream - Sheet1.csv', header=0)
bc_df = pd.read_csv('C:/Users/Алишер/Desktop/NazarbayevUniversity/DatasetPdM/Class_regres_dataset/BC_Upstream_to_Sort - Sheet1.csv', header=0)

# In case there are extra columns, keep only the first three:
ac_df = ac_df.iloc[:, :3]
ca_df = ca_df.iloc[:, :3]
cb_df = cb_df.iloc[:, :3]
bc_df = bc_df.iloc[:, :3]

# Assign column names.
col_names = ['Time series', 'Pressure', 'Fault']
ac_df.columns = col_names
ca_df.columns = col_names
cb_df.columns = col_names
bc_df.columns = col_names

# Add sensor configuration labels.
ac_df['sensor_config'] = 'AC'
ca_df['sensor_config'] = 'CA'
cb_df['sensor_config'] = 'CB'
bc_df['sensor_config'] = 'BC'


In [4]:
# %% [markdown]
"""
## 2. Create Window-Based Aggregated Features

Define a function to convert a time series column into aggregated features over a sliding window.
Aggregated features include: mean, std, min, max, and the last value.
If a label column is provided, its value is taken from the last row of the window.
"""
# %% 
def create_window_features(df, window_size=10, feature_column='Time series', label_column=None):
    windows = []
    labels = []
    for i in range(len(df) - window_size + 1):
        window = df.iloc[i:i+window_size]
        feat_mean = window[feature_column].mean()
        feat_std  = window[feature_column].std()
        feat_min  = window[feature_column].min()
        feat_max  = window[feature_column].max()
        feat_last = window[feature_column].iloc[-1]
        window_feat = {
            f'{feature_column}_mean': feat_mean,
            f'{feature_column}_std': feat_std,
            f'{feature_column}_min': feat_min,
            f'{feature_column}_max': feat_max,
            f'{feature_column}_last': feat_last
        }
        windows.append(window_feat)
        if label_column is not None:
            labels.append(window[label_column].iloc[-1])
    df_window = pd.DataFrame(windows)
    if label_column is not None:
        df_window[label_column] = labels
    return df_window


In [5]:
# %% [markdown]
"""
### Prepare Datasets for Each Task

- **mech_1 Classification:** Combine AC and CA; use Fault for mech_1.
- **mech_2 Classification:** Combine CB and BC; use Fault for mech_2.
- **Pressure Classification:** Combine all files and transform Pressure into a categorical variable.
"""
# %% 
# For mech_1:
df_mech1 = pd.concat([ac_df, ca_df], axis=0).copy()
df_mech1_window = create_window_features(df_mech1, window_size=10, feature_column='Time series', label_column='Fault')
df_mech1_window.rename(columns={'Fault': 'mech_1'}, inplace=True)

# For mech_2:
df_mech2 = pd.concat([cb_df, bc_df], axis=0).copy()
df_mech2_window = create_window_features(df_mech2, window_size=10, feature_column='Time series', label_column='Fault')
df_mech2_window.rename(columns={'Fault': 'mech_2'}, inplace=True)

# For Pressure:
df_pressure = pd.concat([ac_df, ca_df, cb_df, bc_df], axis=0).copy()
df_pressure_window = create_window_features(df_pressure, window_size=10, feature_column='Time series', label_column='Pressure')


In [6]:
# %% [markdown]
"""
### Transform Pressure Target

Convert the Pressure value to categorical:
- 0 ("low"): Pressure < 4.9 bar
- 1 ("high"): Pressure > 5.2 bar
- 2 ("normal"): Pressure between 4.9 and 5.2 bar
"""
# %% 
def transform_pressure(p):
    if p < 4.9:
        return 0   # low
    elif p > 5.2:
        return 1   # high
    else:
        return 2   # normal

df_pressure_window['pressure_class'] = df_pressure_window['Pressure'].apply(transform_pressure)
# Drop the continuous Pressure column as we now use pressure_class.
df_pressure_window = df_pressure_window.drop(columns=['Pressure'])


In [7]:
# %% [markdown]
"""
## 3. Split the Data into Training, Validation, and Test Sets Using Stratification

We now split each task dataset so that the training, validation, and test sets maintain similar class distributions.
"""
# %% 
from sklearn.model_selection import train_test_split

# --- mech_1 Classification ---
X_mech1 = df_mech1_window.drop(columns=['mech_1'])
y_mech1 = df_mech1_window['mech_1']

X_mech1_train, X_mech1_temp, y_mech1_train, y_mech1_temp = train_test_split(
    X_mech1, y_mech1, test_size=0.3, random_state=42, stratify=y_mech1)
X_mech1_val, X_mech1_test, y_mech1_val, y_mech1_test = train_test_split(
    X_mech1_temp, y_mech1_temp, test_size=0.5, random_state=42, stratify=y_mech1_temp)

print("Mech_1 Distribution:")
print("Train:\n", y_mech1_train.value_counts())
print("Validation:\n", y_mech1_val.value_counts())
print("Test:\n", y_mech1_test.value_counts())

# --- mech_2 Classification ---
X_mech2 = df_mech2_window.drop(columns=['mech_2'])
y_mech2 = df_mech2_window['mech_2']

X_mech2_train, X_mech2_temp, y_mech2_train, y_mech2_temp = train_test_split(
    X_mech2, y_mech2, test_size=0.3, random_state=42, stratify=y_mech2)
X_mech2_val, X_mech2_test, y_mech2_val, y_mech2_test = train_test_split(
    X_mech2_temp, y_mech2_temp, test_size=0.5, random_state=42, stratify=y_mech2_temp)

print("\nMech_2 Distribution:")
print("Train:\n", y_mech2_train.value_counts())
print("Validation:\n", y_mech2_val.value_counts())
print("Test:\n", y_mech2_test.value_counts())

# --- Pressure Classification ---
X_pressure = df_pressure_window.drop(columns=['pressure_class'])
y_pressure = df_pressure_window['pressure_class']

X_pressure_train, X_pressure_temp, y_pressure_train, y_pressure_temp = train_test_split(
    X_pressure, y_pressure, test_size=0.3, random_state=42, stratify=y_pressure)
X_pressure_val, X_pressure_test, y_pressure_val, y_pressure_test = train_test_split(
    X_pressure_temp, y_pressure_temp, test_size=0.5, random_state=42, stratify=y_pressure_temp)

print("\nPressure Distribution:")
print("Train:\n", y_pressure_train.value_counts())
print("Validation:\n", y_pressure_val.value_counts())
print("Test:\n", y_pressure_test.value_counts())

# Preprocessing pipelines (without scaling, only imputation)
clf_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])
reg_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# For classification tasks, we transform features using the pipeline.
X_mech1_train_prep = clf_preprocessor.fit_transform(X_mech1_train)
X_mech1_val_prep = clf_preprocessor.transform(X_mech1_val)
X_mech1_test_prep = clf_preprocessor.transform(X_mech1_test)

X_mech2_train_prep = clf_preprocessor.fit_transform(X_mech2_train)
X_mech2_val_prep = clf_preprocessor.transform(X_mech2_val)
X_mech2_test_prep = clf_preprocessor.transform(X_mech2_test)

# For Pressure Classification.
X_pressure_train_prep = reg_preprocessor.fit_transform(X_pressure_train)
X_pressure_val_prep = reg_preprocessor.transform(X_pressure_val)
X_pressure_test_prep = reg_preprocessor.transform(X_pressure_test)


Mech_1 Distribution:
Train:
 mech_1
0    351
1    234
Name: count, dtype: int64
Validation:
 mech_1
0    75
1    50
Name: count, dtype: int64
Test:
 mech_1
0    75
1    51
Name: count, dtype: int64

Mech_2 Distribution:
Train:
 mech_2
0    357
1    227
Name: count, dtype: int64
Validation:
 mech_2
0    77
1    48
Name: count, dtype: int64
Test:
 mech_2
0    77
1    49
Name: count, dtype: int64

Pressure Distribution:
Train:
 pressure_class
2    447
0    425
1    304
Name: count, dtype: int64
Validation:
 pressure_class
2    96
0    91
1    65
Name: count, dtype: int64
Test:
 pressure_class
2    95
0    92
1    65
Name: count, dtype: int64


In [10]:
# %% [markdown]
"""
## 4. Model Training and Evaluation

Train Random Forest classifiers for each task and evaluate on the validation set.
"""
# %% 
# --- mech_1 Classification ---
mech1_clf = RandomForestClassifier(n_estimators=100, random_state=42)
mech1_clf.fit(X_mech1_train_prep, y_mech1_train)
y_mech1_val_pred = mech1_clf.predict(X_mech1_val_prep)
print("Validation results for mech_1 classifier:")
print("Accuracy:", accuracy_score(y_mech1_val, y_mech1_val_pred))
print(classification_report(y_mech1_val, y_mech1_val_pred))

# --- mech_2 Classification ---
mech2_clf = RandomForestClassifier(n_estimators=100, random_state=42)
mech2_clf.fit(X_mech2_train_prep, y_mech2_train)
y_mech2_val_pred = mech2_clf.predict(X_mech2_val_prep)
print("Validation results for mech_2 classifier:")
print("Accuracy:", accuracy_score(y_mech2_val, y_mech2_val_pred))
print(classification_report(y_mech2_val, y_mech2_val_pred))

# --- Pressure Classification ---
pressure_clf = RandomForestClassifier(n_estimators=100, random_state=42)
pressure_clf.fit(X_pressure_train_prep, y_pressure_train)
y_pressure_val_pred = pressure_clf.predict(X_pressure_val_prep)
print("Validation results for pressure classifier:")
print("Accuracy:", accuracy_score(y_pressure_val, y_pressure_val_pred))
print(classification_report(y_pressure_val, y_pressure_val_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_pressure_val, y_pressure_val_pred))


Validation results for mech_1 classifier:
Accuracy: 0.96
              precision    recall  f1-score   support

           0       0.96      0.97      0.97        75
           1       0.96      0.94      0.95        50

    accuracy                           0.96       125
   macro avg       0.96      0.96      0.96       125
weighted avg       0.96      0.96      0.96       125

Validation results for mech_2 classifier:
Accuracy: 0.96
              precision    recall  f1-score   support

           0       0.96      0.97      0.97        77
           1       0.96      0.94      0.95        48

    accuracy                           0.96       125
   macro avg       0.96      0.96      0.96       125
weighted avg       0.96      0.96      0.96       125

Validation results for pressure classifier:
Accuracy: 0.9603174603174603
              precision    recall  f1-score   support

           0       0.95      0.95      0.95        91
           1       0.97      1.00      0.98       

In [11]:
# %% [markdown]
"""
## 5. Deployment Function

Define a function that accepts a window of time series values (list/array) and a sensor configuration.
It computes aggregated features, preprocesses them using the existing pipelines (without scaling), and returns predictions:
- mech_1_prediction (if sensor configuration is AC or CA),
- mech_2_prediction (if sensor configuration is CB or BC),
- pressure_prediction (categorical: 0 for low, 1 for high, 2 for normal).
"""
# %% 
def aggregate_window(window, feature_name='Time series'):
    window = np.array(window)
    return {
        f'{feature_name}_mean': window.mean(),
        f'{feature_name}_std': window.std(),
        f'{feature_name}_min': window.min(),
        f'{feature_name}_max': window.max(),
        f'{feature_name}_last': window[-1]
    }

def predict_system_status(window, sensor_config):
    """
    Parameters:
      - window: List or array of time series values (length must be at least the window size used in training, e.g., 10).
      - sensor_config: A string (one of 'AC', 'CA', 'CB', 'BC').
      
    Returns:
      Dictionary with:
         'mech_1_prediction' (if applicable),
         'mech_2_prediction' (if applicable),
         'pressure_prediction' (categorical: 0 for low, 1 for high, 2 for normal).
    """
    features_dict = aggregate_window(window, feature_name='Time series')
    input_df = pd.DataFrame([features_dict])
    
    # Preprocess aggregated features using the no-scale pipeline.
    input_scaled_clf = clf_preprocessor.transform(input_df)
    input_scaled_reg = reg_preprocessor.transform(input_df)
    
    predictions = {}
    
    if sensor_config.upper() in ['AC', 'CA']:
        pred_mech1 = mech1_clf.predict(input_scaled_clf)[0]
        predictions['mech_1_prediction'] = int(pred_mech1)
    else:
        predictions['mech_1_prediction'] = None
    
    if sensor_config.upper() in ['CB', 'BC']:
        pred_mech2 = mech2_clf.predict(input_scaled_clf)[0]
        predictions['mech_2_prediction'] = int(pred_mech2)
    else:
        predictions['mech_2_prediction'] = None
    
    pred_pressure = pressure_clf.predict(input_scaled_reg)[0]
    predictions['pressure_prediction'] = int(pred_pressure)
    
    return predictions


In [12]:
# %% [markdown]
"""
## 6. Testing the Deployment Function and Test Set Evaluation

Test the deployment function using an example window and sensor configurations.
Also, optionally evaluate the overall test performance for the pressure classifier.
"""
# %% 
example_window = [320, 315, 330, 340, 335, 325, 310, 300, 305, 315]  # Example window (10 datapoints)
for config in ['AC', 'CA', 'CB', 'BC']:
    pred = predict_system_status(example_window, config)
    print(f"Input window (length {len(example_window)}), sensor_config = {config}:")
    print(pred)
    print()

# Optional: Evaluate overall test performance for the pressure classifier.
y_pressure_test_pred = pressure_clf.predict(X_pressure_test_prep)
pressure_test_accuracy = accuracy_score(y_pressure_test, y_pressure_test_pred)
print("Test set pressure classification accuracy:", pressure_test_accuracy)
print("Confusion Matrix:")
print(confusion_matrix(y_pressure_test, y_pressure_test_pred))


Input window (length 10), sensor_config = AC:
{'mech_1_prediction': 0, 'mech_2_prediction': None, 'pressure_prediction': 1}

Input window (length 10), sensor_config = CA:
{'mech_1_prediction': 0, 'mech_2_prediction': None, 'pressure_prediction': 1}

Input window (length 10), sensor_config = CB:
{'mech_1_prediction': None, 'mech_2_prediction': 0, 'pressure_prediction': 1}

Input window (length 10), sensor_config = BC:
{'mech_1_prediction': None, 'mech_2_prediction': 0, 'pressure_prediction': 1}

Test set pressure classification accuracy: 0.9484126984126984
Confusion Matrix:
[[86  0  6]
 [ 1 64  0]
 [ 6  0 89]]


In [14]:
# %% [markdown]
"""
## 6. Testing the Deployment Function and Test Set Evaluation

Test the deployment function using an example window and sensor configurations.
Also, optionally evaluate the overall test performance for the pressure classifier.
"""
# %% 
example_window = [4788,4567,4356,4671,4245,4235]  # Example window (10 datapoints)
for config in ['BC']:
    pred = predict_system_status(example_window, config)
    print(f"Input window (length {len(example_window)}), sensor_config = {config}:")
    print(pred)
    print()

# Optional: Evaluate overall test performance for the pressure classifier.
y_pressure_test_pred = pressure_clf.predict(X_pressure_test_prep)
pressure_test_accuracy = accuracy_score(y_pressure_test, y_pressure_test_pred)
print("Test set pressure classification accuracy:", pressure_test_accuracy)
print("Confusion Matrix:")
print(confusion_matrix(y_pressure_test, y_pressure_test_pred))


Input window (length 6), sensor_config = BC:
{'mech_1_prediction': None, 'mech_2_prediction': 1, 'pressure_prediction': 0}

Test set pressure classification accuracy: 0.9484126984126984
Confusion Matrix:
[[86  0  6]
 [ 1 64  0]
 [ 6  0 89]]
